# 最終大表建置（2018–2024）

這份 notebook 把三個來源合併成建模用的最終大表 `f1_final.csv`。**一列 = 一位車手在一場正賽**，主鍵 `(raceId, driverId)`。

| 來源 | 內容 | 用途 |
|---|---|---|
| `f1_master.csv` | `build_master_2.ipynb` 的輸出 | 主表（決定有哪些列）＋原有欄位 |
| Ergast CSV | `qualifying`、`results`、`lap_times`、`pit_stops`、`status`、`races` | 排位賽表現、開局與超車、進站、可靠度 |
| `f1tel_out/` | `fetch_telemetry_2018_2024.ipynb` 的輸出 | 煞車點、過彎、油門、輪胎、跟車、極速、彎速 |

**三條規則**（所有新欄位都遵守）：

1. **不洩漏結果**：排位賽在正賽之前，排位賽類用**當場**的值；賽道特徵用當站排位賽或過去年份的資料；其他全部用該車手（或車隊）**前 5 場**的平均，不含當場，欄位名稱以 `_l5` 結尾。
2. **扣掉車子影響**：駕駛風格類（煞車點、過彎型態、油門、滑行、輪胎衰退）用「**和隊友的差值**」，欄位名稱含 `_tm`。例如 `brake_point_tm_l5 = 5` 代表前 5 場平均比隊友早 5 公尺煞車。隊友缺資料時為缺值。
3. **車子性能用車隊層級**：極速、加速、彎速取兩位車手中較好的一位，代表車子的上限。

**`f1_master.csv` 不需要重建**：它原本就涵蓋 2011–2024，前 5 場平均也已用更早的比賽算好。這裡直接篩出 2018 年以後，結果和把 `START_YEAR` 改成 2018 重建完全相同。

**這版不含需要人工整理的變數**（動力單元供應商、主場、合約年）。`avg_pit_duration` 由新的 `pit_rel_l5` 取代，不放進最終大表。

## 步驟 0：設定

請在 **f1tif** 環境（kernel）執行，需要 pandas 和 pyarrow。

- `ERGAST_DIR`：14 個 Ergast CSV 所在資料夾。
- `TM_GAP_MAX`：和隊友排位差距超過這個百分比時視為異常（例如排位賽中途下雨、一人撞車只跑出慢圈），設為缺值。預設 5%。
- `TEL_DIR`：抓遙測 notebook 的輸出資料夾 `f1tel_out`。
- `MASTER_CSV`：`f1_master.csv` 的路徑。
- `WINDOW`：前 N 場的 N（預設 5，和 `f1_master` 一致）。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ERGAST_DIR = Path('../raw')                    # 14 個 Ergast CSV 所在資料夾（data/raw/）
TEL_DIR    = Path('../raw/f1tel_out')         # 遙測輸出（data/raw/f1tel_out/）
MASTER_CSV = Path('../processed/f1_master.csv')
OUT_DIR    = Path('../processed')             # f1_final.csv 輸出到 data/processed/
START_YEAR = 2018
HIST_YEAR  = 2011                    # Ergast 類滾動特徵往回算的起點（2018 年初的前 5 場需要 2017 年資料）
WINDOW     = 5
TM_GAP_MAX = 5.0                     # 和隊友排位差距超過 ±5% 視為異常，設為缺值
NA         = ['\\N']

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print('pandas', pd.__version__)

pandas 3.1.1


## 步驟 1：讀取資料

1. **Ergast**：讀需要的 6 張表。`entries` 是每位車手每場正賽一列（2011 年起），附上比賽日期，是後面所有滾動平均的骨架。
2. **主表**：`f1_master.csv` 篩出 2018 年以後。
3. **遙測**：讀 `laps`、`corners`、`lapstats`、`tel`、`id_map`，用 `id_map` 把三字母車手代碼換成 Ergast 的 `raceId`、`driverId`、`constructorId`。`tel` 只讀需要的欄位（跟車圈和排位賽最快圈），控制記憶體用量。

In [2]:
def read_ergast(name):
    return pd.read_csv(ERGAST_DIR / f'{name}.csv', na_values=NA)

races   = read_ergast('races')[['raceId', 'year', 'round', 'circuitId', 'date']]
races['date'] = pd.to_datetime(races['date'])
results = read_ergast('results')
quali   = read_ergast('qualifying')
laptime = read_ergast('lap_times')[['raceId', 'driverId', 'lap', 'position']]
pits    = read_ergast('pit_stops')[['raceId', 'driverId', 'stop', 'lap', 'milliseconds']]
status  = read_ergast('status')

entries = (results[['raceId', 'driverId', 'constructorId', 'grid', 'positionOrder', 'statusId']]
           .merge(races, on='raceId'))
entries = entries[entries['year'] >= HIST_YEAR].sort_values(['date', 'raceId']).reset_index(drop=True)

master = pd.read_csv(MASTER_CSV)
master = master[master['year'] >= START_YEAR].reset_index(drop=True)
print('entries（2011 起）:', len(entries), '| 主表（2018 起）:', len(master))

id_map = pd.read_parquet(TEL_DIR / 'id_map.parquet')
def add_ids(df):
    '''遙測表 + Ergast ID（依 year, round, session, Driver 對應）'''
    return df.merge(id_map[['year', 'round', 'session', 'Driver', 'raceId', 'driverId', 'constructorId']],
                    on=['year', 'round', 'session', 'Driver'], how='inner')

laps     = add_ids(pd.read_parquet(TEL_DIR / 'laps.parquet'))
corners  = add_ids(pd.read_parquet(TEL_DIR / 'corners.parquet'))
lapstats = add_ids(pd.read_parquet(TEL_DIR / 'lapstats.parquet'))
tel = pd.read_parquet(TEL_DIR / 'tel.parquet',
                      columns=['year', 'round', 'session', 'Driver', 'LapNumber', 'sample_type',
                               'Time', 'Speed', 'Throttle', 'Brake', 'DistanceToDriverAhead'])
tel = add_ids(tel[((tel['session'] == 'R') & (tel['sample_type'] == 'follow')) |
                  ((tel['session'] == 'Q') & (tel['sample_type'] == 'fastest'))])
for n, d in [('laps', laps), ('corners', corners), ('lapstats', lapstats), ('tel（篩選後）', tel)]:
    print(f'{n:12s} {len(d):>10,} 列')

entries（2011 起）: 5980 | 主表（2018 起）: 2979
laps            209,051 列
corners         451,628 列
lapstats         27,620 列
tel（篩選後）      9,507,622 列


## 步驟 2：排位賽表現（當場的值）

| 欄位 | 定義 |
|---|---|
| `quali_position` | 排位賽名次（罰退前），取自 `qualifying.position` |
| `grid_penalty` | 起跑位置 − 排位名次；大於 0 代表被罰退。從維修區起跑（`grid = 0`）時，起跑位置以該場最後一位計 |
| `pit_lane_start` | 是否從維修區起跑（0/1） |
| `reached_q3` | 是否進入 Q3（0/1） |
| `teammate_quali_gap_pct` | 和隊友排位圈速的差距 %，正數 = 比隊友慢。比較兩人**都有跑的最後一節**（都進 Q3 比 Q3，否則比 Q2，再否則比 Q1），避免拿 Q1 和 Q3 的圈速比。超過 ±`TM_GAP_MAX`% 的視為異常、設為缺值 |

`quali_gap_pct`、`grid` 已在主表中，維持原樣。

In [3]:
def to_seconds(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    try:
        if ':' in s:
            m, sec = s.split(':')
            return int(m) * 60 + float(sec)
        return float(s)
    except ValueError:
        return np.nan

q = quali[['raceId', 'driverId', 'constructorId', 'position', 'q1', 'q2', 'q3']].copy()
for c in ['q1', 'q2', 'q3']:
    q[c] = q[c].map(to_seconds)
q = q.drop_duplicates(['raceId', 'driverId'])

mate = q.merge(q, on=['raceId', 'constructorId'], suffixes=('', '_m'))
mate = mate[mate['driverId'] != mate['driverId_m']].drop_duplicates(['raceId', 'driverId'])
gap = pd.Series(np.nan, index=mate.index)
for c in ['q1', 'q2', 'q3']:                       # 後面的節次覆蓋前面：兩人都有 Q3 就用 Q3
    both = mate[c].notna() & mate[c + '_m'].notna()
    gap[both] = (mate.loc[both, c] - mate.loc[both, c + '_m']) / mate.loc[both, c + '_m'] * 100
n_out = int((gap.abs() > TM_GAP_MAX).sum())
mate['teammate_quali_gap_pct'] = gap.where(gap.abs() <= TM_GAP_MAX)
print(f'和隊友排位差距超過 ±{TM_GAP_MAX}% 而設為缺值的筆數: {n_out}')

f_quali = q[['raceId', 'driverId', 'position', 'q3']].rename(columns={'position': 'quali_position'})
f_quali['reached_q3'] = f_quali['q3'].notna().astype(int)
f_quali = f_quali.drop(columns='q3').merge(mate[['raceId', 'driverId', 'teammate_quali_gap_pct']],
                                           on=['raceId', 'driverId'], how='left')

g = entries[['raceId', 'driverId', 'grid']].copy()
n_start = g.groupby('raceId')['driverId'].transform('size')
g['pit_lane_start'] = (g['grid'] == 0).astype(int)
g['grid_eff'] = g['grid'].where(g['grid'] > 0, n_start)
f_quali = f_quali.merge(g[['raceId', 'driverId', 'pit_lane_start', 'grid_eff']], on=['raceId', 'driverId'], how='left')
f_quali['grid_penalty'] = f_quali['grid_eff'] - f_quali['quali_position']
f_quali = f_quali.drop(columns='grid_eff')
f_quali.describe().round(2)

和隊友排位差距超過 ±5.0% 而設為缺值的筆數: 105


,raceId,driverId,quali_position,reached_q3,teammate_quali_gap_pct,pit_lane_start,grid_penalty
count,10494.0,10494.00,10494.00,10494.00,10079.00,5956.00,5956.00
mean,624.6,343.00,11.20,0.35,0.00,0.01,0.03
std,428.3,389.59,6.26,0.48,0.89,0.12,1.82
min,1.0,1.00,1.00,0.00,-4.93,0.00,-8.00
25%,160.0,17.00,6.00,0.00,-0.43,0.00,0.00
50%,870.0,60.00,11.00,0.00,0.00,0.00,0.00
75%,1006.0,822.00,16.00,1.00,0.43,0.00,0.00
max,1144.0,862.00,28.00,1.00,4.74,1.00,23.00


## 步驟 3：遙測類的單場數值

先算出「每位車手每場」的原始數值，步驟 6 再取前 5 場平均。

**彎道（來自排位賽最快圈的 `corners`）**：排位賽每人都是全力的一圈，最適合比較煞車和過彎；也因此 2018 巴林正賽缺彎道資料不影響這裡。

- 只用 `apex_found = True` 的彎道。
- 每個彎先算全場中位數，再算每位車手和中位數的差，最後對該站所有彎道取平均，消除不同賽道的差異。
- `brake_point`：需要煞車的彎道，煞車點比全場中位數**早**幾公尺（正數 = 較早煞車）。
- `corner_vratio`：需要煞車的彎道，彎心速度 ÷ 入彎速度的平均。越小越接近 V 型（深煞、低彎心速度），越大越接近 U 型（保持彎速）。
- `low_corner_delta`、`high_corner_delta`：低速彎（全場中位彎心速度 < 120 km/h）和高速彎（≥ 200 km/h）的彎心速度比全場中位數快多少 km/h。這是**車子性能**，取車隊較好的一位。

**油門（來自正賽最快圈的 `lapstats`）**：`full_throttle_pct`、`coast_pct`、`throttle_change_sd`，每人每場取平均。

**極速與加速（來自排位賽）**：

- `top_speed_rel`：車隊排位賽測速點最高速，占該場所有車隊最高者的百分比。
- `accel_delta`：排位賽最快圈中，全油門、車速 150–250 km/h 區段的平均加速度（m/s²），減去該場中位數。取車隊較好的一位。

**和隊友的差值**：`tm_delta()` 算「自己 − 隊友」，駕駛風格類欄位都轉成 `_tm` 版本。

In [4]:
def tm_delta(df, cols):
    '''同一場、同一車隊：自己 − 隊友平均。沒有隊友資料時為缺值'''
    out = df.copy()
    for c in cols:
        grp = out.groupby(['raceId', 'constructorId'])[c]
        s, n = grp.transform('sum'), grp.transform('count')
        other = (s - out[c]) / (n - 1)
        out[c + '_tm'] = (out[c] - other).where((n > 1) & out[c].notna())
    return out

# --- 彎道 ---
if 'apex_found' not in corners.columns:            # 舊版抓取程式沒有這個欄位
    print('⚠️ corners 沒有 apex_found 欄位，全部視為 True')
    corners['apex_found'] = True
corners['apex_found'] = corners['apex_found'].fillna(True)
cq = corners[(corners['session'] == 'Q') & (corners['sample_type'] == 'fastest') & corners['apex_found'].astype(bool)].copy()
key = ['raceId', 'corner']
cq['field_min'] = cq.groupby(key)['min_speed'].transform('median')
cq['speed_delta'] = cq['min_speed'] - cq['field_min']
cq['vratio'] = cq['min_speed'] / cq['entry_speed']
brk = cq[~cq['flat_out'].astype(bool) & cq['brake_dist_m'].notna()].copy()
brk['brake_delta'] = brk['brake_dist_m'] - brk.groupby(key)['brake_dist_m'].transform('median')

drv = ['raceId', 'driverId', 'constructorId']
f_corner = brk.groupby(drv).agg(brake_point=('brake_delta', 'mean'), corner_vratio=('vratio', 'mean')).reset_index()
low  = cq[cq['field_min'] < 120].groupby(drv)['speed_delta'].mean().rename('low_corner_delta')
high = cq[cq['field_min'] >= 200].groupby(drv)['speed_delta'].mean().rename('high_corner_delta')
f_corner = f_corner.merge(low.reset_index(), on=drv, how='outer').merge(high.reset_index(), on=drv, how='outer')

# --- 油門 ---
f_thr = (lapstats[(lapstats['session'] == 'R') & (lapstats['sample_type'] == 'fastest')]
         .groupby(drv)[['full_throttle_pct', 'coast_pct', 'throttle_change_sd']].mean().reset_index())

# --- 極速 ---
lq = laps[laps['session'] == 'Q']
spd = lq.groupby(drv)['SpeedST'].max().reset_index()

# --- 加速度（排位賽最快圈遙測）---
tq = tel[tel['session'] == 'Q'].sort_values(['raceId', 'driverId', 'Time']).copy()
grp = tq.groupby(['raceId', 'driverId'])
dt = grp['Time'].diff()
acc = grp['Speed'].diff() / 3.6 / dt
ok = (tq['Throttle'] >= 98) & tq['Speed'].between(150, 250) & (dt > 0) & (dt < 0.5)
accel = tq.assign(a=acc)[ok].groupby(drv)['a'].mean().rename('accel').reset_index()

f_drv_tel = (f_corner.merge(f_thr, on=drv, how='outer')
                     .merge(spd, on=drv, how='outer')
                     .merge(accel, on=drv, how='outer'))
f_drv_tel = tm_delta(f_drv_tel, ['brake_point', 'corner_vratio', 'full_throttle_pct', 'coast_pct', 'throttle_change_sd'])

# --- 車隊層級的車子性能 ---
team = ['raceId', 'constructorId']
f_car = f_drv_tel.groupby(team).agg(low_corner_delta=('low_corner_delta', 'max'),
                                    high_corner_delta=('high_corner_delta', 'max'),
                                    top_speed=('SpeedST', 'max'),
                                    accel=('accel', 'max')).reset_index()
f_car['top_speed_rel'] = f_car['top_speed'] / f_car.groupby('raceId')['top_speed'].transform('max') * 100
f_car['accel_delta'] = f_car['accel'] - f_car.groupby('raceId')['accel'].transform('median')
f_car = f_car.drop(columns=['top_speed', 'accel'])

print('車手單場遙測特徵:', f_drv_tel.shape, '| 車隊單場車子性能:', f_car.shape)
f_drv_tel.describe().T[['count', 'mean', 'std', 'min', 'max']].round(3)

車手單場遙測特徵: (2978, 17) | 車隊單場車子性能: (1490, 6)


,count,mean,std,min,max
raceId,2978.0,1066.239,45.798,989.000,1144.000
driverId,2978.0,694.831,309.290,1.000,862.000
constructorId,2978.0,81.208,88.021,1.000,215.000
brake_point,2906.0,-0.087,6.291,-59.269,73.901
corner_vratio,2906.0,0.550,0.064,0.316,0.756
low_corner_delta,2907.0,-0.230,3.508,-99.000,11.250
high_corner_delta,2001.0,-0.548,7.538,-131.000,28.000
full_throttle_pct,2832.0,51.764,9.404,12.255,100.000
coast_pct,2832.0,4.815,2.390,0.000,34.174
throttle_change_sd,2832.0,9.139,1.260,0.000,17.892


## 步驟 4：輪胎與跟車（來自正賽 `laps`）

先挑**乾淨圈**：非第 1 圈、非進出站圈、全程綠旗、標記為準確，並把圈速換成「該場乾淨圈中位數的百分比」`lap_pct`（100 = 中位數）。

**輪胎**

- `deg_slope`：同一個 stint 內，`lap_pct` 隨胎齡（`TyreLife`）每圈增加多少百分點，至少要有 6 圈乾淨圈的 stint 才計算，再依圈數加權平均。油量減輕會讓圈速變快、抵銷部分衰退，但同場每台車都一樣，轉成和隊友的差值 `deg_slope_tm` 後會消掉。
- `avg_stint_len`：平均每個 stint 跑幾圈（用全部圈，不只乾淨圈）。

**跟車**（只看乾淨圈，`gap_ahead_s` = 過線時和前車的時間差）

- `follow_gap_med`：和前車差距在 1.5 秒內時，差距的中位數（秒）。越小代表越敢貼近。
- `close_share`：差距 ≤ 1 秒的圈數占比（%）。
- `dirty_air_loss`：差距 ≤ 1 秒的圈平均 `lap_pct`，減去差距 ≥ 2 秒的圈平均 `lap_pct`。正數代表跟車時圈速掉得多。兩種圈各至少 3 圈才計算。

**煞車區貼近度**（來自跟車圈遙測）

- `brake_zone_gap_m`：每次開始踩煞車那一刻和前車的距離，取 0–100 公尺之間的中位數。越小代表進彎時貼得越近。

In [5]:
lr = laps[laps['session'] == 'R'].copy()
clean = (lr['LapTime'].notna() & (lr['LapNumber'] > 1) & lr['PitInTime'].isna() & lr['PitOutTime'].isna()
         & (lr['TrackStatus'].astype(str) == '1'))
if 'IsAccurate' in lr:
    clean &= lr['IsAccurate'].fillna(False).astype(bool)
lc = lr[clean].copy()
lc['lap_pct'] = lc['LapTime'] / lc.groupby('raceId')['LapTime'].transform('median') * 100

# --- 輪胎衰退斜率：每個 stint 一條迴歸線（用加總公式向量化計算）---
st = lc.dropna(subset=['TyreLife', 'Stint']).copy()
st['x'], st['y'] = st['TyreLife'].astype(float), st['lap_pct']
st['xx'], st['xy'] = st['x'] ** 2, st['x'] * st['y']
sk = drv + ['Stint']
agg = st.groupby(sk).agg(n=('x', 'size'), sx=('x', 'sum'), sy=('y', 'sum'), sxx=('xx', 'sum'), sxy=('xy', 'sum')).reset_index()
den = agg['n'] * agg['sxx'] - agg['sx'] ** 2
agg['slope'] = ((agg['n'] * agg['sxy'] - agg['sx'] * agg['sy']) / den).where((agg['n'] >= 6) & (den > 0))
agg = agg.dropna(subset=['slope'])
agg['w'] = agg['slope'] * agg['n']
deg = agg.groupby(drv).agg(w=('w', 'sum'), n=('n', 'sum')).reset_index()
deg['deg_slope'] = deg['w'] / deg['n']
deg = deg[drv + ['deg_slope']]

stint_len = (lr.dropna(subset=['Stint']).groupby(drv + ['Stint']).size()
               .groupby(drv).mean().rename('avg_stint_len').reset_index())

# --- 跟車 ---
gp = lc[lc['gap_ahead_s'] > 0].copy()                 # 排除缺值與異常的負差距
gp['close'] = gp['gap_ahead_s'] <= 1.0
g15 = gp[gp['gap_ahead_s'] <= 1.5].groupby(drv)['gap_ahead_s'].median().rename('follow_gap_med')
share = (gp.groupby(drv)['close'].mean() * 100).rename('close_share')
near = gp[gp['close']].groupby(drv)['lap_pct'].agg(['mean', 'size'])
far  = gp[gp['gap_ahead_s'] >= 2.0].groupby(drv)['lap_pct'].agg(['mean', 'size'])
dirty = near.join(far, lsuffix='_n', rsuffix='_f', how='inner')
dirty = (dirty['mean_n'] - dirty['mean_f']).where((dirty['size_n'] >= 3) & (dirty['size_f'] >= 3)).rename('dirty_air_loss')

# --- 煞車區貼近度（跟車圈遙測）---
tr = tel[tel['session'] == 'R'].sort_values(['raceId', 'driverId', 'LapNumber', 'Time']).copy()
b = tr['Brake'].astype(bool)
onset = b & ~b.groupby([tr['raceId'], tr['driverId'], tr['LapNumber']]).shift(1, fill_value=False)
bz = tr[onset & tr['DistanceToDriverAhead'].between(0, 100, inclusive='neither')]
bz = bz.groupby(drv)['DistanceToDriverAhead'].median().rename('brake_zone_gap_m')

f_race_tel = deg.merge(stint_len, on=drv, how='outer')
for s in [g15, share, dirty, bz]:
    f_race_tel = f_race_tel.merge(s.reset_index(), on=drv, how='outer')
f_race_tel = tm_delta(f_race_tel, ['deg_slope'])
f_race_tel.describe().T[['count', 'mean', 'std', 'min', 'max']].round(3)

/var/folders/35/gxtkvt556z7cp01c2lphdjdw0000gn/T/ipykernel_56485/202403424.py:33: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  dirty = near.join(far, lsuffix='_n', rsuffix='_f', how='inner')


,count,mean,std,min,max
raceId,2970.0,1066.221,45.786,989.000,1144.000
driverId,2970.0,694.434,309.611,1.000,862.000
constructorId,2970.0,81.166,87.997,1.000,215.000
deg_slope,2821.0,-0.038,0.171,-4.586,1.741
avg_stint_len,2970.0,21.421,8.510,1.000,67.000
follow_gap_med,2703.0,0.892,0.238,0.021,1.499
close_share,2805.0,21.342,18.952,0.000,100.000
dirty_air_loss,2074.0,1.187,2.301,-20.143,25.786
brake_zone_gap_m,2488.0,38.140,11.692,0.410,93.184
deg_slope_tm,2694.0,-0.000,0.170,-4.002,4.002


## 步驟 5：開局、超車、進站、可靠度（來自 Ergast，2011 年起）

這些用 Ergast 的每圈名次和進站紀錄，所以可以從 2011 年算起，2018 年初的前 5 場平均不會缺。

| 欄位 | 定義 |
|---|---|
| `lap1_gain` | 起跑位置 − 第 1 圈結束時的名次（正數 = 起步後名次進步）|
| `onroad_gain` | 賽道上的名次淨進步（超車次數 − 被超次數的近似值）。每一圈只比較「當圈和前一圈都沒有進出站」的車之間的相對順序，所以自己或別人進站、別人退賽造成的名次變動都不會算進來 |
| `pit_rel` | 平均進站時間 ÷ 該場有效進站時間中位數。清理規則和 `f1_master` 相同（< 60 秒且 ≤ 中位數 1.5 倍）。消除各賽道維修區長度的差異 |
| `undercut_att`、`undercut_win` | 進站前一圈正前方的車，如果在之後 1–3 圈內也進站，算一次 undercut 嘗試；對方進站後的下一圈，自己在對方前面就算成功。前 5 場以「成功次數 ÷ 嘗試次數」計算成功率 |
| `mech_dnf` | 是否因機械問題未完賽（車隊層級：每場機械故障退賽的車數 ÷ 出賽車數）|

**機械故障的判斷**：未完賽（狀態不是 `Finished`，也不是 `+1 Lap` 這類），且不屬於事故類狀態（`Accident`、`Collision`、`Spun off`、`Disqualified` 等，見 `INCIDENT`）。下方會列出 2018 年以後各種未完賽狀態的次數和分類結果，請檢查是否合理。⚠️ 較新的賽季可能出現籠統的 `Retired`，這裡預設算機械故障，可以視情況修改。

**賽道超車難度** `track_pos_changes_hist`：每場比賽每位車手平均的賽道上名次變動次數（同樣排除進站與退賽造成的變動），對同一賽道「之前年份」取平均，不含當年。

In [6]:
lt = laptime[laptime['raceId'].isin(entries['raceId'])].sort_values(['raceId', 'driverId', 'lap']).copy()

# 自己的進站圈與出站圈
pl = pits[['raceId', 'driverId', 'lap']]
pitlaps = pd.concat([pl, pl.assign(lap=pl['lap'] + 1)]).drop_duplicates().assign(is_pit=True)
lt = lt.merge(pitlaps, on=['raceId', 'driverId', 'lap'], how='left')
lt['prev_pos'] = lt.groupby(['raceId', 'driverId'])['position'].shift(1)
# 只在「當圈與前一圈都沒有進出站」的車之間比較相對名次：排除自己或別人進站、退賽造成的名次變動
ok = lt['is_pit'].isna() & (lt['lap'] >= 2) & lt['prev_pos'].notna()
cl = lt[ok]
rank_prev = cl.groupby(['raceId', 'lap'])['prev_pos'].rank()
rank_cur  = cl.groupby(['raceId', 'lap'])['position'].rank()
lt['delta'] = (rank_prev - rank_cur).reindex(lt.index)

ek = ['raceId', 'driverId']
onroad = lt.groupby(ek)['delta'].sum(min_count=1).rename('onroad_gain').reset_index()
race_chg = (lt.assign(ad=lt['delta'].abs()).groupby('raceId')
              .agg(chg=('ad', 'sum'), n=('driverId', 'nunique')).reset_index())
race_chg['pos_changes'] = race_chg['chg'] / race_chg['n']

# 第 1 圈名次變化
g = entries[['raceId', 'driverId', 'grid']].copy()
g['grid_eff'] = g['grid'].where(g['grid'] > 0, g.groupby('raceId')['driverId'].transform('size'))
lap1 = lt[lt['lap'] == 1][ek + ['position']].merge(g, on=ek)
lap1['lap1_gain'] = lap1['grid_eff'] - lap1['position']

# 進站時間相對值
p = pits[pits['raceId'].isin(entries['raceId'])].copy()
p = p[p['milliseconds'] < 60_000]
p['med'] = p.groupby('raceId')['milliseconds'].transform('median')
p = p[p['milliseconds'] <= 1.5 * p['med']]
p['rel'] = p['milliseconds'] / p['med']
pit_rel = p.groupby(ek)['rel'].mean().rename('pit_rel').reset_index()

# undercut
stops = pits[['raceId', 'driverId', 'lap']].rename(columns={'lap': 'L'})
stops = stops[stops['raceId'].isin(entries['raceId'])]
pos = laptime[['raceId', 'driverId', 'lap', 'position']]
u = stops.assign(lap=stops['L'] - 1).merge(pos, on=['raceId', 'driverId', 'lap']).rename(columns={'position': 'my_pos'})
u = u[u['my_pos'] > 1].assign(position=lambda d: d['my_pos'] - 1)
u = u.merge(pos.rename(columns={'driverId': 'rival'}), on=['raceId', 'lap', 'position'])
rs = stops.rename(columns={'driverId': 'rival', 'L': 'Lr'})
u = u.merge(rs, on=['raceId', 'rival'])
u = u[(u['Lr'] - u['L']).between(1, 3)].sort_values('Lr').drop_duplicates(['raceId', 'driverId', 'L'])
u['lap'] = u['Lr'] + 1
u = (u.merge(pos.rename(columns={'position': 'me_after'}), on=['raceId', 'driverId', 'lap'])
      .merge(pos.rename(columns={'driverId': 'rival', 'position': 'rv_after'}), on=['raceId', 'rival', 'lap']))
u['win'] = (u['me_after'] < u['rv_after']).astype(int)
undercut = u.groupby(ek).agg(undercut_att=('win', 'size'), undercut_win=('win', 'sum')).reset_index()

# 可靠度
INCIDENT = {'Accident', 'Collision', 'Collision damage', 'Spun off', 'Disqualified', 'Withdrew',
            'Did not qualify', 'Did not prequalify', 'Excluded', 'Damage', 'Debris', 'Driver unwell',
            'Illness', 'Injured', 'Injury', 'Not classified', 'Safety concerns'}
st_map = status.set_index('statusId')['status']
e = entries[['raceId', 'driverId', 'constructorId', 'statusId', 'year']].copy()
e['status'] = e['statusId'].map(st_map)
finished = e['status'].eq('Finished') | e['status'].astype(str).str.match(r'^\+\d+ Laps?$')
e['mech_dnf'] = (~finished & ~e['status'].isin(INCIDENT)).astype(int)
team_rel = e.groupby(['raceId', 'constructorId'])['mech_dnf'].mean().rename('mech_dnf_rate').reset_index()

print('2018 年以後的未完賽狀態（請檢查分類）:')
chk = e[(e['year'] >= START_YEAR) & ~finished]
display(chk.groupby(['status', 'mech_dnf']).size().rename('次數').reset_index().sort_values('次數', ascending=False))

f_ergast = (entries[ek].merge(onroad, on=ek, how='left').merge(lap1[ek + ['lap1_gain']], on=ek, how='left')
            .merge(pit_rel, on=ek, how='left').merge(undercut, on=ek, how='left'))
f_ergast[['undercut_att', 'undercut_win']] = f_ergast[['undercut_att', 'undercut_win']].fillna(0)
f_ergast.describe().round(2)

2018 年以後的未完賽狀態（請檢查分類）:


,status,mech_dnf,次數
3,Collision,0,98
0,Accident,0,50
13,Engine,1,46
4,Collision damage,0,42
2,Brakes,1,24
19,Gearbox,1,20
31,Retired,1,20
26,Power Unit,1,17
34,Suspension,1,10
9,Disqualified,0,10


,raceId,driverId,onroad_gain,lap1_gain,pit_rel,undercut_att,undercut_win
count,5980.00,5980.00,5742.00,5808.00,5497.00,5980.00,5980.00
mean,988.39,566.24,0.00,0.26,1.02,0.43,0.10
std,89.69,378.00,3.35,3.02,0.07,0.67,0.33
min,841.00,1.00,-19.00,-23.00,0.67,0.00,0.00
25%,906.00,20.00,-2.00,-1.00,0.98,0.00,0.00
50%,988.00,818.00,0.00,0.00,1.00,0.00,0.00
75%,1066.00,839.00,2.00,2.00,1.03,1.00,0.00
max,1144.00,862.00,19.00,12.00,1.50,4.00,3.00


## 步驟 6：前 5 場平均

用和 `f1_master` 相同的定義：**該車手（或車隊）自己參加的前 5 場正賽**，依比賽日期排序，**不含當場**，缺值的場次自動跳過。

做法：先把每場的原始數值接到 `entries`（每位車手每場一列），依車手分組後 `shift(1)`（往後移一場，排除當場），再取 `rolling(5)` 平均。車隊層級的欄位用同樣方式，依車隊分組。

- 車手層級：駕駛風格 `_tm` 欄位、跟車、輪胎、開局超車、進站。
- 車隊層級：彎速、極速、加速、機械故障率。
- undercut：前 5 場的成功次數總和 ÷ 嘗試次數總和（沒有嘗試時為缺值）。

遙測從 2018 年才有，所以 2018 年頭幾站的遙測類 `_l5` 欄位是用較少場次平均，第 1 站為缺值。

In [7]:
def roll_prev(panel, by, cols, how='mean'):
    '''依 by 分組、日期排序：前 WINDOW 場（不含當場）的平均或總和'''
    panel = panel.sort_values(['date', 'raceId'])
    out = panel[['raceId', by]].copy()
    for c in cols:
        shifted = panel.groupby(by)[c].shift(1)
        r = shifted.groupby(panel[by]).rolling(WINDOW, min_periods=1)
        out[c + '_l5'] = (r.mean() if how == 'mean' else r.sum()).reset_index(level=0, drop=True)
    return out

drv_cols = ['brake_point_tm', 'corner_vratio_tm', 'full_throttle_pct_tm', 'coast_pct_tm', 'throttle_change_sd_tm',
            'deg_slope_tm', 'avg_stint_len', 'follow_gap_med', 'close_share', 'dirty_air_loss', 'brake_zone_gap_m',
            'lap1_gain', 'onroad_gain', 'pit_rel']
dpanel = (entries[['raceId', 'driverId', 'constructorId', 'date']]
          .merge(f_drv_tel.drop(columns='constructorId'), on=ek, how='left')
          .merge(f_race_tel.drop(columns='constructorId'), on=ek, how='left')
          .merge(f_ergast, on=ek, how='left'))
d_l5 = roll_prev(dpanel, 'driverId', drv_cols)

u_l5 = roll_prev(dpanel, 'driverId', ['undercut_att', 'undercut_win'], how='sum')
u_l5['undercut_rate_l5'] = (u_l5['undercut_win_l5'] / u_l5['undercut_att_l5']).where(u_l5['undercut_att_l5'] > 0)
d_l5 = d_l5.merge(u_l5[['raceId', 'driverId', 'undercut_rate_l5']], on=ek)

car_cols = ['low_corner_delta', 'high_corner_delta', 'top_speed_rel', 'accel_delta', 'mech_dnf_rate']
tpanel = (entries[['raceId', 'constructorId', 'date']].drop_duplicates(['raceId', 'constructorId'])
          .merge(f_car, on=['raceId', 'constructorId'], how='left')
          .merge(team_rel, on=['raceId', 'constructorId'], how='left'))
t_l5 = roll_prev(tpanel, 'constructorId', car_cols)

# 賽道特徵（當站可知）
trk = (lapstats[(lapstats['session'] == 'Q') & (lapstats['sample_type'] == 'fastest')]
       .groupby('raceId')['full_throttle_pct'].median().rename('track_full_throttle_pct').reset_index())
rc = race_chg.merge(races[['raceId', 'circuitId', 'date']], on='raceId').sort_values('date')
rc['track_pos_changes_hist'] = rc.groupby('circuitId')['pos_changes'].transform(lambda s: s.shift(1).expanding().mean())
f_track = trk.merge(rc[['raceId', 'track_pos_changes_hist']], on='raceId', how='outer')
print('車手滾動:', d_l5.shape, '| 車隊滾動:', t_l5.shape, '| 賽道:', f_track.shape)

車手滾動: (5980, 17) | 車隊滾動: (2991, 7) | 賽道: (286, 3)


## 步驟 7：合併成最終大表

以主表為準，全部用 `LEFT JOIN`，保證主表的列不會消失或重複：

- 排位賽表現：`(raceId, driverId)`
- 車手滾動特徵：`(raceId, driverId)`
- 車隊滾動特徵：`(raceId, constructorId)`
- 賽道特徵：`raceId`

欄位依變數清單的類別排列，`podium`（目標變數）放最後。

In [8]:
final = (master.drop(columns=['avg_pit_duration'])
         .merge(f_quali, on=ek, how='left')
         .merge(d_l5, on=ek, how='left')
         .merge(t_l5, on=['raceId', 'constructorId'], how='left')
         .merge(f_track, on='raceId', how='left'))

ORDER = {
    '識別':     ['raceId', 'year', 'circuitId', 'driverId', 'constructorId'],
    '排位賽表現': ['grid', 'quali_gap_pct', 'quali_position', 'grid_penalty', 'pit_lane_start', 'reached_q3',
                'teammate_quali_gap_pct'],
    '車子性能':   ['con_points_l5', 'top_speed_rel_l5', 'accel_delta_l5', 'low_corner_delta_l5',
                'high_corner_delta_l5', 'mech_dnf_rate_l5'],
    '駕駛行為':   ['brake_point_tm_l5', 'corner_vratio_tm_l5', 'full_throttle_pct_tm_l5', 'coast_pct_tm_l5',
                'throttle_change_sd_tm_l5', 'lap_time_mean', 'lap_time_sd', 'deg_slope_tm_l5', 'avg_stint_len_l5',
                'lap1_gain_l5', 'onroad_gain_l5', 'n_pit_stops', 'pit_rel_l5', 'undercut_rate_l5',
                'follow_gap_med_l5', 'close_share_l5', 'dirty_air_loss_l5', 'brake_zone_gap_m_l5'],
    '近期表現':   ['drv_avg_finish_l5'],
    '情境':       ['street_circuit', 'track_full_throttle_pct', 'track_pos_changes_hist', 'air_temp_mean', 'precip_mm'],
    '風格群':     ['style_cluster'],
    '目標':       ['podium'],
}
cols = [c for group in ORDER.values() for c in group]
missing = [c for c in cols if c not in final.columns]
extra = [c for c in final.columns if c not in cols]
assert not missing, f'缺少欄位: {missing}'
final = final[cols]
print(final.shape, '| 未列入排序的欄位:', extra)
final.head()

(2979, 44) | 未列入排序的欄位: []


,raceId,year,circuitId,driverId,constructorId,grid,quali_gap_pct,quali_position,grid_penalty,pit_lane_start,reached_q3,teammate_quali_gap_pct,con_points_l5,top_speed_rel_l5,accel_delta_l5,low_corner_delta_l5,high_corner_delta_l5,mech_dnf_rate_l5,brake_point_tm_l5,corner_vratio_tm_l5,full_throttle_pct_tm_l5,coast_pct_tm_l5,throttle_change_sd_tm_l5,lap_time_mean,lap_time_sd,deg_slope_tm_l5,avg_stint_len_l5,lap1_gain_l5,onroad_gain_l5,n_pit_stops,pit_rel_l5,undercut_rate_l5,follow_gap_med_l5,close_share_l5,dirty_air_loss_l5,brake_zone_gap_m_l5,drv_avg_finish_l5,street_circuit,track_full_throttle_pct,track_pos_changes_hist,air_temp_mean,precip_mm,style_cluster,podium
0,989,2018,1,20,6,3,0.830417,3.0,0.0,0.0,1.0,0.012221,27.4,NaN,NaN,NaN,NaN,0.1,NaN,NaN,NaN,NaN,NaN,98.716213,1.027951,NaN,NaN,-1.2,-0.4,2.25,1.044388,0.0,NaN,NaN,NaN,NaN,5.8,0,66.279904,2.698592,22.6,0.0,NaN,1
1,989,2018,1,1,131,1,0.000000,1.0,0.0,0.0,1.0,-0.046291,33.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,98.695903,0.970204,NaN,NaN,-2.8,4.4,1.80,1.025030,1.0,NaN,NaN,NaN,NaN,3.4,0,66.279904,2.698592,22.6,0.0,NaN,1
2,989,2018,1,8,6,2,0.818097,2.0,0.0,0.0,1.0,-0.012219,27.4,NaN,NaN,NaN,NaN,0.1,NaN,NaN,NaN,NaN,NaN,98.900870,0.938434,NaN,NaN,-2.0,2.0,1.60,1.000659,0.0,NaN,NaN,NaN,NaN,3.6,0,66.279904,2.698592,22.6,0.0,NaN,1
3,989,2018,1,817,9,8,1.217288,5.0,3.0,0.0,1.0,0.333419,19.6,NaN,NaN,NaN,NaN,0.3,NaN,NaN,NaN,NaN,NaN,99.212246,0.812067,NaN,NaN,0.0,1.0,2.50,1.047070,0.0,NaN,NaN,NaN,NaN,13.4,0,66.279904,2.698592,22.6,0.0,NaN,0
4,989,2018,1,4,1,10,2.997634,11.0,-1.0,0.0,0.0,-0.192003,1.4,NaN,NaN,NaN,NaN,0.1,NaN,NaN,NaN,NaN,NaN,100.418540,1.031992,NaN,NaN,1.4,-1.6,1.75,1.015767,0.5,NaN,NaN,NaN,NaN,11.0,0,66.279904,2.698592,22.6,0.0,NaN,0


## 步驟 8：品質檢查

1. **列數與主鍵**：最終大表列數必須等於主表 2018 年以後的列數，`(raceId, driverId)` 不重複。
2. **資料洩漏**：任選一位車手，手動取他前 5 場的 `lap1_gain` 平均，和大表的 `lap1_gain_l5` 比對。
3. **缺值**：列出每個欄位的缺值比例。預期的缺值來源：
   - 遙測類 `_l5`：2018 年第 1 站（沒有前一場）、車手生涯初期。
   - `_tm` 欄位：隊友那幾場缺資料。
   - 彎道類：2020 年 Mugello 和薩基爾外圈（沒有彎道資料）。
   - `undercut_rate_l5`：前 5 場都沒有 undercut 嘗試。
   - `style_cluster`：尚待建模組員分群後接入，預期全部為缺值。

In [9]:
assert len(final) == len(master), '列數不一致：某個 join 造成重複或遺失'
assert not final.duplicated(['raceId', 'driverId']).any(), '主鍵重複'
print('✓ 列數與主表一致:', len(final), '| ✓ (raceId, driverId) 無重複')

# 洩漏檢查：取 2019 年有出賽的第一位車手
d0 = final.loc[final['year'] == 2019, 'driverId'].iloc[0]
hist = dpanel[dpanel['driverId'] == d0].sort_values(['date', 'raceId']).reset_index(drop=True)
for rid in final.loc[(final['driverId'] == d0) & (final['year'] == 2019), 'raceId'].head(3):
    i = hist.index[hist['raceId'] == rid][0]
    manual = hist.iloc[max(0, i - WINDOW):i]['lap1_gain'].mean()
    in_table = final.loc[(final['raceId'] == rid) & (final['driverId'] == d0), 'lap1_gain_l5'].item()
    print(f'driverId {d0} raceId {rid}: 手算 {manual:.3f}，大表 {in_table:.3f}',
          '✓' if np.isclose(manual, in_table, equal_nan=True) else '✗')

summary = pd.DataFrame({'缺值比例': final.isna().mean().round(3)}).join(
    final.describe().T[['mean', 'std', 'min', 'max']].round(3))
summary

✓ 列數與主表一致: 2979 | ✓ (raceId, driverId) 無重複
driverId 822 raceId 1010: 手算 0.200，大表 0.200 ✓
driverId 822 raceId 1011: 手算 0.400，大表 0.400 ✓
driverId 822 raceId 1012: 手算 0.800，大表 0.800 ✓


,缺值比例,mean,std,min,max
raceId,0.000,1066.216,45.808,989.000,1144.000
year,0.000,2021.106,2.030,2018.000,2024.000
circuitId,0.000,30.392,27.437,1.000,80.000
driverId,0.000,694.881,309.250,1.000,862.000
constructorId,0.000,81.182,88.018,1.000,215.000
grid,0.000,10.064,5.821,0.000,20.000
quali_gap_pct,0.013,2.006,2.255,0.000,45.596
quali_position,0.001,10.488,5.761,1.000,20.000
grid_penalty,0.001,0.037,1.880,-8.000,18.000
pit_lane_start,0.001,0.023,0.151,0.000,1.000


## 步驟 9：輸出

存成 `f1_final.csv`（UTF-8，不含索引欄），缺值在 CSV 中是空白。

In [10]:
out_path = OUT_DIR / 'f1_final.csv'
final.to_csv(out_path, index=False)
print('已輸出:', out_path.resolve(), final.shape)

已輸出: /Users/xinyuhuang/Desktop/資料科學/project/f1_final.csv (2979, 44)


## 欄位定義總表

「時間」欄：**當場** = 該場比賽開始前已知；**前 5 場** = 該車手／車隊前 5 場平均，不含當場；**歷史** = 同賽道過去年份平均。

| 類別 | 欄位 | 定義 | 單位 | 時間 | 層級 |
|---|---|---|---|---|---|
| 識別 | `raceId`, `year`, `circuitId`, `driverId`, `constructorId` | Ergast 編號 | — | — | — |
| 排位賽表現 | `grid` | 正賽起跑位置；0 = 維修區起跑 | 名次 | 當場 | 車手 |
| 排位賽表現 | `quali_gap_pct` | 最佳排位圈比全場最快圈慢的 % | % | 當場 | 車手 |
| 排位賽表現 | `quali_position` | 排位賽名次（罰退前） | 名次 | 當場 | 車手 |
| 排位賽表現 | `grid_penalty` | 起跑位置 − 排位名次；> 0 = 被罰退 | 名次 | 當場 | 車手 |
| 排位賽表現 | `pit_lane_start` | 是否從維修區起跑 | 0/1 | 當場 | 車手 |
| 排位賽表現 | `reached_q3` | 是否進入 Q3 | 0/1 | 當場 | 車手 |
| 排位賽表現 | `teammate_quali_gap_pct` | 和隊友在兩人都有跑的最後一節的圈速差；正 = 較慢 | % | 當場 | 車手 |
| 車子性能 | `con_points_l5` | 車隊兩台車合計積分平均 | 分 | 前 5 場 | 車隊 |
| 車子性能 | `top_speed_rel_l5` | 排位賽測速點極速占該場最快車隊的 % | % | 前 5 場 | 車隊 |
| 車子性能 | `accel_delta_l5` | 全油門 150–250 km/h 平均加速度 − 該場中位數 | m/s² | 前 5 場 | 車隊 |
| 車子性能 | `low_corner_delta_l5` | 低速彎（< 120 km/h）彎心速度 − 全場中位數 | km/h | 前 5 場 | 車隊 |
| 車子性能 | `high_corner_delta_l5` | 高速彎（≥ 200 km/h）彎心速度 − 全場中位數 | km/h | 前 5 場 | 車隊 |
| 車子性能 | `mech_dnf_rate_l5` | 機械故障退賽車數 ÷ 出賽車數 | 比例 | 前 5 場 | 車隊 |
| 駕駛行為 | `brake_point_tm_l5` | 煞車點比全場中位數早幾公尺，再減隊友；正 = 比隊友早煞 | 公尺 | 前 5 場 | 車手 |
| 駕駛行為 | `corner_vratio_tm_l5` | 彎心速度 ÷ 入彎速度，減隊友；負 = 比隊友更 V 型 | 比值 | 前 5 場 | 車手 |
| 駕駛行為 | `full_throttle_pct_tm_l5` | 全油門時間占比，減隊友 | 百分點 | 前 5 場 | 車手 |
| 駕駛行為 | `coast_pct_tm_l5` | 滑行時間占比，減隊友 | 百分點 | 前 5 場 | 車手 |
| 駕駛行為 | `throttle_change_sd_tm_l5` | 油門變化量標準差，減隊友；正 = 比隊友不平順 | % | 前 5 場 | 車手 |
| 駕駛行為 | `lap_time_mean`, `lap_time_sd` | 乾淨圈速與穩定度（原主表欄位） | % of race median | 前 5 場 | 車手 |
| 駕駛行為 | `deg_slope_tm_l5` | stint 內每圈圈速增加的百分點，減隊友；正 = 比隊友吃胎 | 百分點／圈 | 前 5 場 | 車手 |
| 駕駛行為 | `avg_stint_len_l5` | 平均 stint 圈數 | 圈 | 前 5 場 | 車手 |
| 駕駛行為 | `lap1_gain_l5` | 第 1 圈名次進步 | 名次 | 前 5 場 | 車手 |
| 駕駛行為 | `onroad_gain_l5` | 賽道上的名次淨進步（排除進站、退賽造成的變動） | 名次 | 前 5 場 | 車手 |
| 駕駛行為 | `n_pit_stops` | 平均有效進站次數（原主表欄位） | 次 | 前 5 場 | 車手 |
| 駕駛行為 | `pit_rel_l5` | 進站時間 ÷ 該場中位數 | 比值 | 前 5 場 | 車手 |
| 駕駛行為 | `undercut_rate_l5` | undercut 成功次數 ÷ 嘗試次數 | 比例 | 前 5 場 | 車手 |
| 駕駛行為 | `follow_gap_med_l5` | 1.5 秒內跟車時的差距中位數 | 秒 | 前 5 場 | 車手 |
| 駕駛行為 | `close_share_l5` | 和前車 ≤ 1 秒的乾淨圈占比 | % | 前 5 場 | 車手 |
| 駕駛行為 | `dirty_air_loss_l5` | ≤ 1 秒跟車圈與 ≥ 2 秒圈的圈速差 | 百分點 | 前 5 場 | 車手 |
| 駕駛行為 | `brake_zone_gap_m_l5` | 開始煞車時和前車的距離中位數 | 公尺 | 前 5 場 | 車手 |
| 近期表現 | `drv_avg_finish_l5` | 平均最終排序（原主表欄位） | 名次 | 前 5 場 | 車手 |
| 情境 | `street_circuit` | 街道賽（原主表欄位） | 0/1 | 當場 | 賽道 |
| 情境 | `track_full_throttle_pct` | 當站排位賽最快圈全油門占比的中位數 | % | 當場 | 賽道 |
| 情境 | `track_pos_changes_hist` | 同賽道過去年份每位車手的平均名次變動次數 | 次 | 歷史 | 賽道 |
| 情境 | `air_temp_mean`, `precip_mm` | 比賽時段氣溫與降雨（原主表欄位） | °C、mm | 當場（預報） | 賽道 |
| 風格群 | `style_cluster` | 駕駛風格分群結果（待接入） | 類別 | — | 車手 |
| 目標 | `podium` | 最終排序前三名 = 1 | 0/1 | 結果 | 車手 |

**主要限制**

1. 遙測資料來自 TracingInsights（經 tif1 取得），每秒約 7–8 筆；煞車只有開／關，沒有力道。
2. 遙測類特徵只取抽樣圈：排位賽每人最快一圈；正賽每人最快 5 圈與最多 5 圈跟車圈。
3. `onroad_gain` 以相鄰兩圈的相對順序近似超車，同一圈內超車又被超回會互相抵消。
4. 機械故障的分類依賴 Ergast 狀態文字，籠統的 `Retired` 預設算機械故障。
5. 2020 年 Mugello 和薩基爾外圈沒有彎道資料，彎道類特徵在之後 5 場的平均中會少算這一場。
6. `style_cluster` 由建模組員分群後接入；分群模型應只用訓練期資料 fit，避免洩漏。